<a href="https://colab.research.google.com/github/mannduuu07-png/urban-fire-risk-analysiskorea-fire-frequency-severity-analysis/blob/main/notebooks/04_robustness_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - Robustness Checks

Tests whether the frequency-severity relationship is stable across
different sampling thresholds and time periods.

In [1]:
import pandas as pd
from scipy.stats import spearmanr

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

df = pd.read_parquet('/content/drive/MyDrive/fire_data/processed/cleaned_fire_data.parquet')

def region_stats_excl_worst(data, min_count=100):
    result = []
    for (sido, sigungu), group in data.groupby(['시도', '시군구']):
        total_count = len(group)
        if total_count < min_count:
            continue
        total_casualty = group['인명피해(명)소계'].sum()
        worst_idx = group['인명피해(명)소계'].idxmax()
        worst_val = group.loc[worst_idx, '인명피해(명)소계']
        casualty_excl = total_casualty - worst_val
        result.append({
            '시도': sido, '시군구': sigungu, '화재건수': total_count,
            'casualties_per_100_incl': total_casualty / total_count * 100,
            'casualties_per_100_excl': casualty_excl / (total_count - 1) * 100,
            'worst_incident_casualties': worst_val,
        })
    return pd.DataFrame(result)

def top_n_tuples(region_df, col, n=10):
    top = region_df.nlargest(n, col)
    return set(zip(top['시도'], top['시군구']))

Mounted at /content/drive


## Threshold sensitivity (200 / 300 / 500 cumulative fires)

In [2]:
for min_c in [200, 300, 500]:
    r = region_stats_excl_worst(df, min_count=min_c)
    c, p = spearmanr(r['화재건수'], r['casualties_per_100_excl'])
    ov = len(top_n_tuples(r, '화재건수') & top_n_tuples(r, 'casualties_per_100_excl'))
    print(f"min_count={min_c}: n_regions={len(r)}, rho={c:.3f}(p={p:.4f}), top10_overlap={ov}")

min_count=200: n_regions=276, rho=0.130(p=0.0304), top10_overlap=0
min_count=300: n_regions=263, rho=0.100(p=0.1071), top10_overlap=0
min_count=500: n_regions=237, rho=0.108(p=0.0964), top10_overlap=0


## Period-split persistence: 2015-2019 vs 2020-2024

### Bug found and fixed during this analysis
An earlier version compared top-10 lists using district name alone
(e.g. `'중구'`). This is wrong: 중구, 남구, 동구, 서구 etc. are
shared by multiple cities nationwide (중구 alone exists in 6
different cities). Comparing by name caused two *different* cities
to be counted as "the same persistent region." Fixed by comparing
`(시도, 시군구)` tuples instead.

In [3]:
periods = {'2015-2019': range(2015, 2020), '2020-2024': range(2020, 2025)}
top10_by_period = {}
for name, yrs in periods.items():
    sub = df[df['연도'].isin(yrs)]
    r = region_stats_excl_worst(sub, min_count=150)
    c, p = spearmanr(r['화재건수'], r['casualties_per_100_excl'])
    t10 = top_n_tuples(r, 'casualties_per_100_excl')
    top10_by_period[name] = t10
    print(f"{name}: n_regions={len(r)}, rho={c:.3f}(p={p:.4f})")
    print(f"  High-severity top10: {t10}")

persist = top10_by_period['2015-2019'] & top10_by_period['2020-2024']
print(f"\nRegions in high-severity top10 in BOTH periods: {len(persist)} -- {persist}")

2015-2019: n_regions=249, rho=0.047(p=0.4581)
  High-severity top10: {('경기도', '안양시동안구'), ('강원도', '삼척시'), ('충청북도', '청주시상당구'), ('경상북도', '안동시'), ('대구광역시', '중구'), ('경상북도', '청송군'), ('대구광역시', '남구'), ('경기도', '고양시일산동구'), ('경기도', '수원시팔달구'), ('경상북도', '포항시북구')}
2020-2024: n_regions=262, rho=0.212(p=0.0006)
  High-severity top10: {('인천광역시', '남동구'), ('서울특별시', '동대문구'), ('울산광역시', '중구'), ('충청북도', '청주시상당구'), ('충청북도', '제천시'), ('경기도', '구리시'), ('인천광역시', '미추홀구'), ('충청북도', '진천군'), ('강원특별자치도', '영월군'), ('부산광역시', '사상구')}

Regions in high-severity top10 in BOTH periods: 1 -- {('충청북도', '청주시상당구')}
